In [2]:
import numpy as np
from pymoo.indicators.hv import HV
import pickle
import statistics
import pandas as pd

# Hypervolume:

In [3]:
experiment_names = ['zdt1'] # ['zdt1', 'zdt2', 'zdt3', 'zdt4', 'zdt6'] # Name of the experiment
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]
choose_crowd_distance_type = [0] # [0]

# Name of chosen files
file_names = [
        f"E{i+1}V{j+1}D{k+1}C{l+1}_{experiment_name}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for l in choose_crowd_distance_type
        for experiment_name in experiment_names
    ]
num_dataset = len(file_names) # Number of dataset

# Take the results
results = []
for i in range(num_dataset):
  with open("result/" + file_names[i], "rb") as f:
    results.append(pickle.load(f))

### Calculate the Hypervolume:

In [4]:
# Create the hypervolume indicator
ref_point = np.array([11, 11])
indicator = HV(ref_point=ref_point)

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Min. HV', 'Max. HV', 'Combined HV']
# Store the datas
data = []
for i, fn in enumerate(file_names):
  HVs = []
  r = results[i]
  for j in range(len(results[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  if len(HVs) > 1:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), statistics.stdev(HVs), min(HVs), max(HVs), indicator(results[i]['combined'][1])])
  else:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), 0, min(HVs), max(HVs), indicator(results[i]['combined'][1])])

# Create the DataFrame
df = pd.DataFrame(data, columns=columns)
df

,Algorithm,Function,Mean HV,Std. Dev.,Min. HV,Max. HV,Combined HV
0,E1V1D1C1,ZDT1,120.659358,0,120.659358,120.659358,120.659358
1,E1V1D2C1,ZDT1,120.659370,0,120.659370,120.659370,120.659370
2,E1V1D3C1,ZDT1,120.657466,0,120.657466,120.657466,120.657466
3,E1V1D4C1,ZDT1,120.661120,0,120.661120,120.661120,120.661120
4,E1V1D5C1,ZDT1,120.661033,0,120.661033,120.661033,120.661033
